In [ ]:
import pickle
import os
import sys

dir2 = os.path.abspath("")
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path:
    sys.path.append(dir1)
os.chdir("../..")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import shap
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder

In [ ]:
df = pd.read_csv(os.path.join("notebooks", "feature_importance", "data.csv"), index_col=False)

In [ ]:
encoder = TargetEncoder()
df['Model_Encoded'] = encoder.fit_transform(df['Method'], df['Value'])

In [ ]:
df['Rank'] = df.groupby('Dataset')['Value'].rank(ascending=False)

In [ ]:
df[df['Dataset'] == 'brightkite'][['Dataset', 'Method', 'Value', 'Rank', 'Model_Encoded']]

In [ ]:
# Split by dataset_name to ensure entire groups are in train or test, not split between them
train_datasets, test_datasets = train_test_split(df['Dataset'].unique(), test_size=0.1, random_state=142)

train_df = df[df['Dataset'].isin(train_datasets)]
test_df = df[df['Dataset'].isin(test_datasets)]

In [ ]:
X_train = train_df[train_df.columns.difference(['Dataset', 'Dataset_group', 'Method', 'Value', 'Rank'])]
y_train = train_df['Rank'].values
group_train = train_df.groupby('Dataset')['Dataset'].count().to_numpy()

X_test = test_df[test_df.columns.difference(['Dataset', 'Dataset_group', 'Method', 'Value', 'Rank'])]
y_test = test_df['Rank'].values
group_test = test_df.groupby('Dataset')['Dataset'].count().to_numpy()

train_data = lgb.Dataset(X_train, label=y_train, group=group_train, free_raw_data=False)
test_data = lgb.Dataset(X_test, label=y_test, group=group_test, reference=train_data, free_raw_data=False)

In [ ]:
train_df.keys()

In [ ]:
train_df[['Dataset', 'Nu', 'Ni', 'Nr', 'SpaceSize', 'Shape','Value', 'Model_Encoded', 'Rank',]]

In [ ]:
param = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [1, 2, 3, 5],
    'boosting_type': 'gbdt',
    'num_leaves': 41,
    'learning_rate': 0.001,
    'bagging_fraction': 0.8
}

num_round = 1000
bst = lgb.train(param, train_data, num_round, valid_sets=[test_data], early_stopping_rounds=10, verbose_eval=30)

In [ ]:
# Predict the score for test data
train_df['Predicted_Score'] = bst.predict(X_train, num_iteration=bst.best_iteration)

# Rank the predictions
train_df['Predicted_Rank']= train_df.groupby('Dataset')['Predicted_Score'].rank(ascending=False, method='first')


In [ ]:
test_df.keys()

In [ ]:
train_df[['Rank', 'Predicted_Rank']]

In [ ]:
import scipy.stats
# Calculate Mean Spearman Rank Correlation Coefficient:
spearman_corr_list = []
num_of_methods = train_df['Method'].nunique()
y_pred = pd.DataFrame(bst.predict(X_train, num_iteration=bst.best_iteration)).rank(ascending=False, method='first')
for i in range(len(y_pred) // num_of_methods):
    spearman_corr, _ = scipy.stats.spearmanr(train_df['Rank'][i*num_of_methods:(i+1)*num_of_methods], y_pred[i*num_of_methods:(i+1)*num_of_methods])
    spearman_corr_list.append(spearman_corr)
    
print("Mean Spearman Rank Correlation Coefficient:", np.mean(spearman_corr_list))

In [ ]:
# Use SHAP to interpret the model
explainer = shap.Explainer(bst)
shap_values = explainer(X_train)

In [ ]:
shap.summary_plot(shap_values, X_train, plot_type="bar")

In [ ]:
# Plot feature importance
shap.summary_plot(shap_values, X_train)

In [ ]:
mean_shap_values = np.abs(shap_values.values).mean(axis=0)
feature_importance = pd.Series(mean_shap_values, index=X_test.columns).sort_values(ascending=False)
print(feature_importance)

In [ ]:
# Predict the score for test data
train_df['Predicted_Score'] = bst.predict(X_train, num_iteration=bst.best_iteration)

# Rank the predictions
train_df['Predicted_Rank'] = train_df.groupby('Dataset')['Predicted_Score'].rank(ascending=False, method='first')

In [ ]:
import scipy.stats
# Calculate Mean Spearman Rank Correlation Coefficient:
spearman_corr_list = []
num_of_methods = train_df['Method'].nunique()
y_pred = train_df['Predicted_Rank']
for i in range(len(y_pred) // num_of_methods):
    spearman_corr, _ = scipy.stats.spearmanr(train_df['Rank'][i*num_of_methods:(i+1)*num_of_methods], y_pred[i*num_of_methods:(i+1)*num_of_methods])
    spearman_corr_list.append(spearman_corr)
    
print("Mean Spearman Rank Correlation Coefficient:", np.mean(spearman_corr_list))
